# 5 · The best tile size

The browser does not upscale a frame in one go: it cuts it into **tiles**, runs the
model on each, and joins the results. This notebook trains nothing. It takes the model
from `3_upscale_web.ipynb` and asks:

1. **How big should a tile be?**
2. **How many tiles per run (batch)?**
3. **Which upscale factor - x2, x3 or x4?**
4. **What still works on a weaker GPU?**

### Three words used throughout

- **step** - the LR pixels a tile contributes to the picture.
- **halo** - extra pixels around the step that the model needs to see (4 here). The
  model input is `step + 2 x halo` on each side; the halo is computed and thrown away.
- **computed / kept / delivered** pixels - what the GPU computes, what a tile keeps, and
  what the viewer gets. `computed / kept` is the *halo tax*. If the tile does not divide
  the frame, the last row and column hang over the edge: that waste is the *tiling tax*.

Frames per second cannot compare two tile sizes (a bigger tile *is* more of a frame),
so speed is measured in **nanoseconds per output pixel**. At 1080p and 60 fps the whole
budget is 8.0 ns per pixel.

Needs: `checkpoints/upscale_web.pt` from `3_upscale_web.ipynb`.
Results: `summary.md`, "The geometry a client runs at".

In [ ]:
"""Settings."""

import itertools
import math

import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F

from helpers import models, speed, tiles, training, webexport

SWEEP_DIR = training.CHECKPOINT_DIR / "geometry"   # the many test graphs (not published)

STEPS = (32, 64, 96, 128, 160, 192, 256, 320, 384, 512)   # square steps to time
BATCHES = (1, 2, 4, 8, 16)
BATCH_STEPS = (64, 128)
SCALES = (2, 3, 4)

FRAMES = {"720p": (1280, 720), "1080p": (1920, 1080), "1440p": (2560, 1440), "4K": (3840, 2160)}
FRAME = FRAMES["1080p"]
BUDGET_NS_PER_PIXEL = 16.7e6 / (FRAME[0] * FRAME[1])   # 60 fps at 1080p

# Measured on the benchmark page: NVIDIA Ampere, Chromium, WebGPU, fp16, GPU-resident,
# graph capture. Treat differences under 10% as noise.
BROWSER_MS = [
    {"scale": 2, "step": (64, 64), "batch": 1, "ms": 0.186},
    {"scale": 2, "step": (128, 128), "batch": 1, "ms": 0.293},
    {"scale": 2, "step": (192, 192), "batch": 1, "ms": 0.516},
    {"scale": 2, "step": (256, 256), "batch": 1, "ms": 0.893},
    {"scale": 2, "step": (384, 384), "batch": 1, "ms": 1.721},
    {"scale": 2, "step": (135, 192), "batch": 1, "ms": 0.420},   # exact 1080p tilings
    {"scale": 2, "step": (270, 192), "batch": 1, "ms": 0.789},
    {"scale": 2, "step": (180, 320), "batch": 1, "ms": 0.755},
    {"scale": 2, "step": (270, 320), "batch": 1, "ms": 1.175},
    {"scale": 2, "step": (128, 128), "batch": 4, "ms": 0.850},   # batches
    {"scale": 2, "step": (128, 128), "batch": 16, "ms": 3.040},
    {"scale": 2, "step": (540, 960), "batch": 1, "ms": 5.703},   # the whole frame in one run
    {"scale": 3, "step": (128, 128), "batch": 1, "ms": 0.446},   # other scales
    {"scale": 4, "step": (128, 128), "batch": 1, "ms": 0.463},
]
# The same card in an older, quieter run (summary.md): ms per tile by square step.
SUMMARY_TILE_MS = {64: 0.155, 128: 0.202, 192: 0.311, 256: 0.452, 320: 0.657}

# Rough vendor figures (fp32 TFLOPS, GB/s) - used for ratios only.
GPUS = {
    "Iris Xe 96EU": {"tflops": 1.7, "tier": "mid-low, shared memory"},
    "GTX 1650": {"tflops": 2.9, "tier": "mid-low"},
    "Apple M1 8-core": {"tflops": 2.6, "tier": "mid-low, shared memory"},
    "RTX 3050 laptop": {"tflops": 5.5, "tier": "mid"},
    "RX 6600": {"tflops": 8.9, "tier": "mid"},
    "RTX 3060": {"tflops": 12.7, "tier": "mid-high (assumed for the Ampere numbers)"},
}
REFERENCE_GPU = "RTX 3060"

# WebGPU's default limits: a session cannot use bigger buffers unless the page asks.
MAX_BUFFER_BYTES = 256 * 1024 * 1024
MAX_BINDING_BYTES = 128 * 1024 * 1024

# Published to dist/upscale/ at the end, for the browser to measure.
PUBLISH_STEPS = (64, 128, 192, 256, 384)
PUBLISH_BATCH_STEP = 128
PUBLISH_BATCHES = (4, 16)
PUBLISH_SCALES = (3, 4)
PUBLISH_EXACT_KEPT = ((270, 384), (540, 384), (360, 640), (540, 640))   # (height, width) kept

print(f"frame budget: {BUDGET_NS_PER_PIXEL:.2f} ns per output pixel at 1080p60")

In [ ]:
"""The trained model from 3_upscale_web.ipynb."""

checkpoint = training.load_checkpoint(training.CHECKPOINT_DIR / "upscale_web.pt", "cpu")


def build(scale=2):
    """The model at x2, x3 or x4. At x2 all trained weights load. At x3/x4 the skip is
    computed exactly but the tail has another shape and stays untrained (zero): good for
    timing, says nothing about quality."""
    model = models.PlainUpscaler(tuple(checkpoint["widths"]), checkpoint["stem"], checkpoint["skip"], scale)
    own = model.state_dict()
    fitting = {k: v for k, v in checkpoint["state_dict"].items() if k in own and own[k].shape == v.shape}
    model.load_state_dict(fitting, strict=False)
    return model.eval()


model = build(2)
HALO = model.halo
print(f"widths {checkpoint['widths']}, halo {HALO}, {models.count_parameters(model):,} parameters, "
      f"trained {checkpoint['trained']}")


def geometry(step, scale=2, batch=1):
    return tiles.geometry(step, HALO, scale, batch)


def shape_text(pair):
    """(height, width) as 'WxH'."""
    return f"{pair[1]}x{pair[0]}"

## 1. Halo tax against tiling tax

A bigger tile wastes less on its halo, but a tile that does not divide the frame wastes
its overhang. The total is what a 1080p frame computes for every pixel it delivers.

In [ ]:
print(f"{'step':>6}{'input':>10}{'kept':>10}{'halo tax':>10}{'tiles':>7}{'tiling tax':>12}{'total':>8}")
for step in STEPS:
    shape = geometry(step)
    cover = tiles.tiling(FRAME, shape["kept_hr"])
    print(f"{step:>6}{shape_text(shape['input'][2:]):>10}{shape_text(shape['kept_hr']):>10}"
          f"{shape['halo_tax']:>9.3f}x{cover['tiles']:>7}{cover['tiling_tax']:>11.3f}x"
          f"{shape['halo_tax'] * cover['tiling_tax']:>7.3f}x")

## 2. Tiles do not have to be square

No square tile divides 1920x1080 at a useful size, but rectangles do: 384x270, 640x360,
640x540... **640x360 divides every 16:9 frame** (720p is 2x2, 1080p 3x3, 1440p 4x4,
4K 6x6), so one graph serves them all. `tiles.plan_tiling` picks the exact tiling
nearest the target size.

In [ ]:
for name, frame in FRAMES.items():
    exact = [row for row in tiles.exact_tilings(frame, halo=HALO) if 128 ** 2 <= row["kept_px"] <= 640 ** 2]
    plan = tiles.plan_tiling(frame, halo=HALO)
    print(f"{name:>6}: {len(exact):3} exact tilings of a useful size; "
          f"chosen {shape_text(plan['kept_hr'])} out, {plan['cols']}x{plan['rows']} tiles, "
          f"input {shape_text(plan['input'][2:])}{'  (padded)' if plan['padded'] else ''}")

### Tiling must not change the picture

With a halo as wide as the model's reach, a frame built from tiles is identical to the
frame run whole. One pixel less halo, and seams appear.

In [ ]:
def upscale_tiled(model, frame_lr, step, halo, scale=2):
    """Run an LR frame tile by tile and join the kept centres."""
    step_h, step_w = tiles.as_size(step)
    padded = F.pad(frame_lr, (halo,) * 4, mode="replicate")
    height, width = frame_lr.shape[2:]
    out = torch.zeros(1, 3, height * scale, width * scale)
    for top in range(0, height, step_h):
        for left in range(0, width, step_w):
            tile = padded[:, :, top:top + step_h + 2 * halo, left:left + step_w + 2 * halo]
            with torch.no_grad():
                result = model(tile)
            out[:, :, top * scale:(top + step_h) * scale, left * scale:(left + step_w) * scale] = \
                result[:, :, halo * scale:(halo + step_h) * scale, halo * scale:(halo + step_w) * scale]
    return out


plan = tiles.plan_tiling(FRAME, halo=HALO)
step_h, step_w = plan["step"]
frame_lr = torch.rand(1, 3, step_h * 2, step_w * 2)
with torch.no_grad():
    whole = model(frame_lr)
edge = 2 * HALO * 2   # the true frame border may differ, so compare the inside
for halo in (HALO, HALO - 1):
    tiled = upscale_tiled(model, frame_lr, (step_h, step_w), halo)
    difference = float((tiled - whole)[..., edge:-edge, edge:-edge].abs().max())
    print(f"halo {halo}: max difference {difference:.1e}  "
          f"{'identical' if difference < 1e-5 else 'SEAMS - halo too small'}")

## 3. A run = a fixed cost + a cost per pixel

```
ms per run = a + b x computed pixels
```

`a` is paid once per run whatever the size (issuing the call, binding buffers, one GPU
submit). `b` is the model's work per pixel. Big `a` favours big tiles. A weaker GPU has a
bigger `b`. We time square steps on this CPU, then fit `a` and `b` for the CPU and for
the browser numbers above.

In [ ]:
def measure(step, batch=1, scale=2, frame=FRAME):
    """Export one geometry (once), time it on the CPU, and price a whole frame with it."""
    shape = geometry(step, scale, batch)
    step_h, step_w = shape["step"]
    name = f"geom_x{scale}_s{step_w}x{step_h}_b{batch}.onnx"
    if not (SWEEP_DIR / name).exists():
        webexport.export(model if scale == 2 else build(scale), name, shape["input"][2:],
                         halo=HALO, batch=batch, models_dir=SWEEP_DIR)
    ms = webexport.time_graph(SWEEP_DIR / name, shape["input"][2:], batch=batch)
    cover = tiles.tiling(frame, shape["kept_hr"], batch)
    return {**shape, "ms": ms, "runs": cover["runs"], "frame_ms": ms * cover["runs"],
            "ns_per_kept": ms * 1e6 / shape["kept"],
            "ns_per_delivered": ms * cover["runs"] * 1e6 / cover["delivered"]}


sweep = [measure(step) for step in STEPS]
print(f"{'step':>6}{'ms/run':>9}{'ns/kept px':>12}{'ns/frame px':>13}{'runs':>6}{'frame ms':>10}")
for row in sweep:
    print(f"{row['step'][0]:>6}{row['ms']:>9.3f}{row['ns_per_kept']:>12.2f}{row['ns_per_delivered']:>13.2f}"
          f"{row['runs']:>6}{row['frame_ms']:>10.1f}")

In [ ]:
def fit(pairs):
    """Least squares line through (computed pixels, ms): returns a (ms) and b (ns per pixel)."""
    pixels, ms = np.array(pairs, dtype=np.float64).T
    slope, intercept = np.polyfit(pixels, ms, 1)
    return {"fixed_ms": float(intercept), "ns_per_computed": float(slope) * 1e6}


def run_ms(shape, machine):
    """Predicted ms for one run of a geometry on a fitted machine."""
    return machine["fixed_ms"] + machine["ns_per_computed"] * shape["computed"] / 1e6


browser = [{**row, **geometry(row["step"], row["scale"], row["batch"])} for row in BROWSER_MS]
fits = {
    "this CPU": fit([(row["computed"], row["ms"]) for row in sweep]),
    "Ampere (browser)": fit([(row["computed"], row["ms"]) for row in browser
                             if row["scale"] == 2 and row["batch"] == 1]),
    "Ampere (older run)": fit([(geometry(step)["computed"], ms) for step, ms in SUMMARY_TILE_MS.items()]),
}
ampere = fits["Ampere (browser)"]
for name, machine in fits.items():
    print(f"{name:<20} a = {machine['fixed_ms']:.4f} ms per run, b = {machine['ns_per_computed']:.4f} ns per pixel")

# the two Ampere runs are the same card on different days: absolute numbers are a band this wide
spread = ampere["ns_per_computed"] / fits["Ampere (older run)"]["ns_per_computed"]
crossover = ampere["fixed_ms"] * 1e6 / ampere["ns_per_computed"]
print(f"\nOn the Ampere card one run's fixed cost equals {crossover:,.0f} computed pixels "
      f"(a step of about {math.sqrt(crossover) / 2:.0f} LR pixels). Smaller tiles are mostly overhead.")

In [ ]:
"""The browser measurements, priced per pixel."""

print(f"{'scale':>5}{'step':>10}{'keeps':>10}{'batch':>6}{'ms/run':>8}{'ns/kept':>9}{'ns/frame':>10}{'runs':>6}{'1080p':>9}")
for row in browser:
    cover = tiles.tiling(FRAME, row["kept_hr"], row["batch"])
    row.update(runs=cover["runs"], frame_ms=row["ms"] * cover["runs"],
               ns_per_kept=row["ms"] * 1e6 / row["kept"],
               ns_per_delivered=row["ms"] * cover["runs"] * 1e6 / cover["delivered"])
for row in sorted(browser, key=lambda row: (row["scale"], row["ns_per_delivered"])):
    print(f"x{row['scale']:<4}{shape_text(row['step']):>10}{shape_text(row['kept_hr']):>10}{row['batch']:>6}"
          f"{row['ms']:>8.3f}{row['ns_per_kept']:>9.2f}{row['ns_per_delivered']:>10.2f}{row['runs']:>6}"
          f"{row['frame_ms']:>6.1f} ms")

tiled = min((r for r in browser if r["scale"] == 2 and r["runs"] > 1), key=lambda r: r["ns_per_delivered"])
whole_frame = min((r for r in browser if r["runs"] == 1), key=lambda r: r["ns_per_delivered"])
print(f"\ncheapest tiling: {shape_text(tiled['kept_hr'])} at {tiled['ns_per_delivered']:.2f} ns/px "
      f"({tiled['frame_ms']:.1f} ms a frame, {tiled['runs']} runs); whole frame: {whole_frame['ns_per_delivered']:.2f} ns/px")

### The best step on each GPU

We model a weaker GPU as the same fixed cost `a` and a bigger `b` (scaled by its
TFLOPS). Crude, but enough to see the shape.

In [ ]:
def frame_cost(step, machine, frame=FRAME):
    """ns per delivered pixel for a 1080p frame, from a fitted machine."""
    shape = geometry(step)
    cover = tiles.tiling(frame, shape["kept_hr"])
    return run_ms(shape, machine) * cover["runs"] * 1e6 / cover["delivered"]


machines = {"Ampere (measured)": ampere}
for name, spec in GPUS.items():
    machines[f"{name} (scaled)"] = {"fixed_ms": ampere["fixed_ms"],
                                    "ns_per_computed": ampere["ns_per_computed"]
                                    * GPUS[REFERENCE_GPU]["tflops"] / spec["tflops"]}

fine_steps = list(range(32, 561, 8))
print(f"{'machine':<26}{'best step':>10}{'ns/px':>8}{'at plan':>9}{'whole':>8}{'of budget':>12}")
for name, machine in machines.items():
    cost, step = min((frame_cost(step, machine), step) for step in fine_steps)
    at_plan = frame_cost(plan["step"], machine)
    print(f"{name:<26}{step:>10}{cost:>8.2f}{at_plan:>9.2f}{frame_cost((540, 960), machine):>8.2f}"
          f"{at_plan / spread / BUDGET_NS_PER_PIXEL:>6.0%}-{at_plan / BUDGET_NS_PER_PIXEL:.0%}")

plt.figure(figsize=(9, 4))
for name, machine in machines.items():
    plt.plot(fine_steps, [frame_cost(step, machine) for step in fine_steps], label=name)
plt.axhline(BUDGET_NS_PER_PIXEL, color="black", linestyle="--", linewidth=1, label="1080p60 budget")
plt.axvline(plan["step"][1], color="grey", linestyle=":", label=f"{shape_text(plan['kept_hr'])} tile")
plt.xlabel("LR step (pixels)")
plt.ylabel("ns per delivered pixel")
plt.yscale("log")
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()

Bigger is always cheaper on this model: the cheapest is one run for the whole frame.
But a remote desktop usually changes in a small area, and a tile that did not change can
be skipped. A bigger tile skips less. So the limit on tile size is not speed; it is
**what can be skipped** (section 5), **what fits in memory** (section 6), and keeping the
GPU free for the rest of the page.

## 4. Batching

A batch (N tiles in one run) also saves fixed costs, like a bigger tile. It pays the halo
N times, but keeps small tiles, so unchanged tiles can still be skipped.

In [ ]:
batch_rows = [measure(step, batch) for step, batch in itertools.product(BATCH_STEPS, BATCHES)]
print(f"{'step':>6}{'batch':>6}{'ms/run':>9}{'ns/kept px':>12}")
for row in batch_rows:
    print(f"{row['step'][0]:>6}{row['batch']:>6}{row['ms']:>9.3f}{row['ns_per_kept']:>12.2f}")

measured = {r["batch"]: r["ms"] for r in browser if r["scale"] == 2 and r["step"] == (PUBLISH_BATCH_STEP,) * 2}
print(f"\nAmpere, step {PUBLISH_BATCH_STEP}:  {'batch':>5}{'predicted':>11}{'measured':>10}{'ms/tile':>9}")
for batch in BATCHES:
    predicted = run_ms(geometry(PUBLISH_BATCH_STEP, batch=batch), ampere)
    seen = measured.get(batch)
    print(f"{'':<20}{batch:>5}{predicted:>11.3f}{(f'{seen:.3f}' if seen else '-'):>10}"
          f"{(seen or predicted) / batch:>9.3f}")

## 5. Only some of the screen changes

On a desktop, a moving cursor or a line of typed text changes a small area. Only the
tiles it touches need to run. The best tile size then depends on *what* moved.

In [ ]:
CONTENT = {"cursor, 32x32": (32, 32), "a line of text, 400x30": (400, 30),
           "a dragged window, 800x600": (800, 600), "video, 1280x720": (1280, 720), "everything": FRAME}
DIRTY_STEPS = (64, 128, 192, 256, 320, 384)


def dirty_runs(kept_hr, region):
    """Average tiles touched by a `region` (w, h) at a random position."""
    kept_h, kept_w = kept_hr
    cover = tiles.tiling(FRAME, kept_hr)
    return math.ceil(min(cover["cols"], region[0] / kept_w + 1) * min(cover["rows"], region[1] / kept_h + 1))


print(f"{'what changed':<26}" + "".join(f"{step:>8}" for step in DIRTY_STEPS) + "   (ms per frame, Ampere)")
for name, region in CONTENT.items():
    cells = [run_ms(geometry(step), ampere) * dirty_runs(geometry(step)["kept_hr"], region) for step in DIRTY_STEPS]
    print(f"{name:<26}" + "".join(f"{ms:>8.2f}" for ms in cells))

whole_ms = run_ms(geometry((FRAME[1] // 2, FRAME[0] // 2)), ampere)
tiles_ms = plan["runs"] * run_ms(geometry(plan["step"]), ampere)
print(f"\nwhole frame in one run: {whole_ms:.2f} ms; all {plan['runs']} tiles of "
      f"{shape_text(plan['kept_hr'])}: {tiles_ms:.2f} ms")
print(f"so tiling wins while fewer than {whole_ms / tiles_ms:.0%} of tiles change")

## 6. Memory and weaker GPUs

The widest tensor is the 32-channel body at the input size. It grows with tile area and
batch, and WebGPU has default buffer limits (128 MiB per binding, 256 MiB per buffer).

In [ ]:
def working_set(step, batch=1, scale=2, widest=32, bytes_per_value=2):
    """The largest tensor of one fp16 run, and all of them together, in bytes."""
    _, _, height, width = geometry(step, scale, batch)["input"]
    tensors = [batch * channels * height * width * bytes_per_value
               for channels in (3, widest, 3 * scale * scale)] + [batch * 3 * height * width * scale * scale * bytes_per_value]
    return max(tensors), sum(tensors)


print(f"{'step':>6}{'batch':>6}{'largest':>12}{'total':>12}")
for step, batch in itertools.product((128, 256, 512), (1, 4, 16)):
    largest, total = working_set(step, batch)
    flags = ("  over the binding limit" if largest > MAX_BINDING_BYTES else "") + \
            ("  over the buffer limit" if total > MAX_BUFFER_BYTES else "")
    print(f"{step:>6}{batch:>6}{largest / 2**20:>8.1f} MiB{total / 2**20:>8.1f} MiB{flags}")

macs_out = speed.macs_per_lr_pixel(model) / 4
reference_floor = 2 * macs_out / (GPUS[REFERENCE_GPU]["tflops"] * 1e12) * 1e9   # ns per pixel at peak
utilisation = reference_floor / ampere["ns_per_computed"]
print(f"\n{macs_out:,.0f} MACs per output pixel; the Ampere card reaches about {utilisation:.0%} of its peak")
print(f"{'card':<18}{'tier':<42}{'expected ns/px':>16}{'of 1080p60 budget':>20}")
for name, spec in GPUS.items():
    floor = 2 * macs_out / (spec["tflops"] * 1e12) * 1e9
    fast, slow = floor / (utilisation * spread), floor / utilisation
    print(f"{name:<18}{spec['tier']:<42}{fast:>7.2f} -{slow:>6.2f}"
          f"{fast / BUDGET_NS_PER_PIXEL:>12.0%} -{slow / BUDGET_NS_PER_PIXEL:.0%}")

## 7. The upscale factor

All convolutions run at low resolution, so the work per *output* pixel falls with the
square of the scale: x4 does a quarter of x2's work per output pixel, and the stream
sends a quarter of the pixels. The cost is quality - and the data here is x2 only, so the
x3/x4 models are untrained and **no quality is claimed** for them.

In [ ]:
print(f"{'scale':>6}{'MACs/out px':>13}{'CPU ms/run':>12}{'browser ms/run':>16}{'browser ns/frame px':>21}")
for scale in SCALES:
    row = measure(128, scale=scale)
    seen = next((r for r in browser if r["scale"] == scale and r["step"] == (128, 128) and r["batch"] == 1), None)
    print(f"x{scale:<5}{speed.macs_per_lr_pixel(build(scale)) / scale ** 2:>13,.0f}{row['ms']:>12.3f}"
          f"{seen['ms'] if seen else float('nan'):>16.3f}{seen['ns_per_delivered'] if seen else float('nan'):>21.2f}")

## 8. Publish the family

Every geometry as an fp16 graph in `dist/upscale/`, for the benchmark page to measure on
real GPUs. Put new browser numbers back into `BROWSER_MS` above and re-run.

In [ ]:
def publish(step, batch=1, scale=2, note=""):
    step_h, step_w = tiles.as_size(step)
    label = f"geom x{scale} step {step_w}x{step_h}" + (f" batch {batch}" if batch > 1 else "") + \
            (f" - {note}" if note else "")
    name = f"upscale_geom_x{scale}_s{step_w:04d}x{step_h:04d}_b{batch:02d}.onnx"
    fp32 = webexport.export(model if scale == 2 else build(scale), name, (step_h + 2 * HALO, step_w + 2 * HALO),
                            label=f"{label} fp32", halo=HALO, batch=batch, models_dir=SWEEP_DIR)
    fp16 = webexport.to_float16(fp32, name)
    webexport.write_metadata(fp16["path"], {"label": f"{label} fp16"})
    print(webexport.summary_line(fp16))


for step in PUBLISH_STEPS:
    publish(step)
for batch in PUBLISH_BATCHES:
    publish(PUBLISH_BATCH_STEP, batch=batch)
for scale in PUBLISH_SCALES:
    publish(128, scale=scale)
for kept in PUBLISH_EXACT_KEPT:
    cover = tiles.tiling(FRAME, kept)
    publish((kept[0] // 2, kept[1] // 2), note=f"exact 1080p, {cover['cols']}x{cover['rows']} tiles")

print("\nmeasure them in a browser: uv run src/upscale/benchmark/main.py")

## The answer

**640x360 output tiles: a 320x180 step, a 328x188 model input.** It was the cheapest
tiling measured (about 6.8 ms for a 1080p frame in 9 runs), and it divides every 16:9
resolution exactly. `tiles.TARGET_KEPT_PX` is set to it.

- The old 128 step costs about 70% more per frame: 40 runs, each with a fixed cost.
- Rectangles that fit the frame beat squares that do not.
- One run for the whole frame is cheaper still, but cannot skip unchanged tiles.
- Batching helps, less than predicted.
- x4 is half the cost per output pixel of x2, but its quality is not measured yet.
- On a mid-low GPU, plan for 720p60 or 1080p30.

The full reasoning and numbers are in `summary.md`.